# 02 · Limpeza e Padronização

**Objetivo:** ler os brutos, padronizar tipos e harmonizar RAIS×CAGED num schema comum.

**Entradas:** `data/raw/...`.  
**Saídas:** `data/interim/rais.parquet` e `data/interim/caged.parquet`.

**Decisões:** motivos crus → categorias unificadas; escolaridade/idade em faixas canônicas.  
**Limitações:** pequenas incompatibilidades de categorização entre fontes são aproximadas;
registros inconsistentes são *logados e quantificados*, não descartados em silêncio.

In [ ]:
# Preâmbulo: torna o pacote src importável a partir do notebook
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src.config import load_config, anos_validos
cfg = load_config()
print('Raiz:', cfg['root'], '| modo sintético:', cfg['synthetic_mode'])

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq
from src import io_utils, cleaning
raw, interim = cfg['abs']['raw'], cfg['abs']['interim']
out_root = interim / 'rais'   # interim particionado: rais/ano=YYYY/<fonte>.parquet
out_root.mkdir(parents=True, exist_ok=True)
ufs = cfg.get('ufs_subset')

def escreve_particao(dest, chunks):
    '''Escreve chunks limpos num parquet via ParquetWriter (memória baixa).'''
    writer = None; n = 0
    for ch in chunks:
        t = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(dest, t.schema)
        writer.write_table(t); n += len(ch)
    if writer: writer.close()
    return n

In [ ]:
# Ingestão incremental -> interim canônico particionado por ano.
# Modo REAL: extrai cada .7z, lê em chunks, limpa, grava parquet e APAGA o
# .COMT/.txt (poupa disco). Idempotente: pula partições já gravadas.
for ano in cfg['anos']:
    pdir = out_root / f'ano={ano}'
    pdir.mkdir(parents=True, exist_ok=True)
    if cfg['synthetic_mode']:
        for pq_in in sorted((raw / 'RAIS' / str(ano)).glob('*.parquet')):
            dest = pdir / 'synthetic.parquet'
            if dest.exists(): continue
            rais = cleaning.clean_rais(pd.read_parquet(pq_in))
            n = escreve_particao(dest, [rais]); print(f'{ano} sintético: {n:,}')
    else:
        for z in sorted((raw/'RAIS'/str(ano)).glob('*.7z')):
            regiao = z.stem.replace('RAIS_VINC_PUB_', '')
            dest = pdir / f'{regiao}.parquet'
            if dest.exists(): print(f'{ano}/{regiao}: já processado'); continue
            print(f'{ano}/{regiao}: extraindo ...', flush=True)
            extraidos = io_utils.extract_7z(z, raw/'RAIS'/str(ano))
            arq = next(p for p in extraidos if p.suffix.upper() in ('.COMT', '.TXT'))
            print(f'{ano}/{regiao}: lendo+limpando {arq.name} ...', flush=True)
            n = escreve_particao(dest, cleaning.iter_rais_clean_chunks(arq, ano, ufs))
            arq.unlink()   # libera o extraído (mantém o .7z como cache de download)
            print(f'{ano}/{regiao}: {n:,} vínculos -> {dest.name} (extraído removido)')
print('Interim particionado em', out_root)

In [ ]:
# Conferência: linhas e distribuição de motivos sobre o interim (amostra leve)
files = sorted(out_root.rglob('*.parquet'))
print('partições:', len(files))
tot = 0; vc = None
for f in files:
    s = pd.read_parquet(f, columns=['motivo_unificado'])
    tot += len(s)
    vc = s['motivo_unificado'].value_counts() if vc is None else vc.add(
         s['motivo_unificado'].value_counts(), fill_value=0)
print('Total de vínculos:', f'{tot:,}')
display((vc / vc.sum()).round(4).sort_values(ascending=False))